# 08 Split Into One Word Per Row

Reads `processed/07_stopwords.csv` and writes one row per person, cue and word, numbered in the order the person listed them ("bag doritos chips" gives three rows, positions 1, 2, 3).

- `position` counts the words that are left after the earlier steps ("has four legs" becomes `four` = 1, `leg` = 2).
- If a person writes the same word twice in one answer, only the first one is kept. `times_in_answer` says how many times it appeared.
- Answers with no words left are not in the file; they are listed in `processed/08_empty_answers.csv`.

Columns:

| Column | Meaning |
|---|---|
| `row_id` | row in `raw_data_all/all_data.csv` (links back to the original answer and source file) |
| `subject_id` | person (unique across all data) |
| `cue` | the cue word |
| `position` | order of the word in that person's answer for that cue |
| `word` | the word after all processing (lemma) |
| `original_word` | the word as the person typed it (after spell check and abbreviations) |
| `pos` | part of speech |
| `times_in_answer` | how many times the word appeared in the answer |

Output: `processed/08_words.csv`.

In [1]:
import pandas as pd
from collections import Counter

In [2]:
data = pd.read_csv("processed/07_stopwords.csv", dtype=str, keep_default_na=False)
print(f"{len(data):,} answers")

185,465 answers


In [3]:
rows = []
empty = []

for row_id, subject_id, cue, tokens in zip(data["row_id"], data["subject_id"], data["cue"], data["tokens"]):
    items = [item.rsplit("|", 2) for item in tokens.split()]
    if not items:
        empty.append((row_id, subject_id, cue))
        continue
    times = Counter(lemma for _, lemma, _ in items)
    seen = set()
    position = 0
    for text, lemma, pos in items:
        if lemma in seen:
            continue
        seen.add(lemma)
        position += 1
        rows.append((row_id, subject_id, cue, position, lemma, text, pos, times[lemma]))

words = pd.DataFrame(rows, columns=["row_id", "subject_id", "cue", "position", "word",
                                    "original_word", "pos", "times_in_answer"])
print(f"{len(words):,} rows, {len(empty):,} answers with no words left")

953,005 rows, 1,081 answers with no words left


In [4]:
pd.DataFrame(empty, columns=["row_id", "subject_id", "cue"]).to_csv("processed/08_empty_answers.csv", index=False)
words.to_csv("processed/08_words.csv", index=False)
print(f"Saved {len(words):,} rows to processed/08_words.csv")

Saved 953,005 rows to processed/08_words.csv


## Checks

In [5]:
print(f"{words['subject_id'].nunique():,} people, {words['cue'].nunique():,} cues, {words['word'].nunique():,} different words")
print(f"words per answer: mean {len(words) / words[['row_id']].nunique().iloc[0]:.1f}")
print(f"repeated words removed: {(words['times_in_answer'] - 1).sum():,}")
words.groupby("cue").size().describe()

13,630 people, 3,722 cues, 29,901 different words
words per answer: mean 5.2
repeated words removed: 33,551


count    3722.000000
mean      256.046480
std       181.695404
min        54.000000
25%       131.000000
50%       180.000000
75%       329.000000
max      2267.000000
dtype: float64

In [6]:
words[words["cue"] == "sidewalk"].head(12)

,row_id,subject_id,cue,position,word,original_word,pos,times_in_answer
255487,57393,1072,sidewalk,1,road,road,PROPN,1
255488,57393,1072,sidewalk,2,pedestrian,pedestrian,NOUN,1
255489,57393,1072,sidewalk,3,cement,cement,NOUN,1
255511,57398,1123,sidewalk,1,walk,walking,VERB,1
255512,57398,1123,sidewalk,2,next,next,ADV,1
255513,57398,1123,sidewalk,3,road,roads,NOUN,1
255514,57398,1123,sidewalk,4,good,good,ADJ,1
255515,57398,1123,sidewalk,5,pedestrian,pedestrians,NOUN,1
255516,57398,1123,sidewalk,6,often,often,ADV,1
255517,57398,1123,sidewalk,7,concrete,concrete,NOUN,1


In [7]:
words["word"].value_counts().head(25)

word
people       6651
person       5111
one          4533
good         4471
food         4198
time         3948
place        3721
water        3601
small        3220
animal       3110
part         2973
money        2792
bad          2712
work         2640
hard         2565
color        2563
different    2472
body         2457
long         2437
many         2405
large        2292
take         2264
type         2175
big          2156
come         2129
Name: count, dtype: int64